In [ ]:
# ============================================================
# RETO 3 - INTEGRACIÓN DE DATOS
# US HEALTH INSURANCE - K-MEDOIDS (PAM)
# ============================================================

import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score


# ============================================================
# 1. DESCARGAR Y CARGAR LA BASE DE DATOS
# ============================================================

path = kagglehub.dataset_download(
    "teertha/ushealthinsurancedataset"
)
XDB = pd.read_csv(ruta_csv)

print("Archivo cargado:", os.path.basename(ruta_csv))
print("Dimensiones:", XDB.shape)

display(XDB.head())


# ============================================================
# 2. PREPARAR VARIABLES DE ENTRADA
# ============================================================

# Variables que se utilizan para formar los clusters
variables_entrada = [
    "age",
    "sex",
    "bmi",
    "children",
    "smoker"
]

# Variables que analizaremos DESPUÉS de crear los clusters:
# region y charges

XDB = XDB.dropna(
    subset=variables_entrada + ["region", "charges"]
).copy()

XD = XDB[variables_entrada].copy()

# Convertir variables categóricas a numéricas
XD = pd.get_dummies(
    XD,
    columns=["sex", "smoker"],
    dtype=int
)

# Estandarizar
scaler = StandardScaler()

XD_escalado = scaler.fit_transform(XD)

print("\nVARIABLES PARA K-MEDOIDS")
display(XD.head())


# ============================================================
# 3. EVALUAR EL NÚMERO DE CLUSTERS
# ============================================================

resultados = []

for k in range(2, 8):

    modelo = KMedoids(
        n_clusters=k,
        metric="euclidean",
        method="pam",
        init="k-medoids++",
        random_state=42
    )

    clusters = modelo.fit_predict(XD_escalado)

    silueta = silhouette_score(
        XD_escalado,
        clusters,
        sample_size=min(1000, len(XD_escalado)),
        random_state=42
    )

    resultados.append({
        "K": k,
        "Silueta": silueta,
        "Costo_PAM": modelo.inertia_
    })

tabla_k = pd.DataFrame(resultados)

print("\nEVALUACIÓN DEL NÚMERO DE CLUSTERS")
display(tabla_k.round(4))

# Gráfico de silueta
tabla_k.plot(
    x="K",
    y="Silueta",
    kind="line",
    marker="o",
    legend=False,
    figsize=(8, 4),
    title="Selección del número de clusters"
)

plt.xlabel("Número de clusters")
plt.ylabel("Coeficiente de silueta")
plt.tight_layout()
plt.show()


# ============================================================
# 4. MODELO DEFINITIVO K-MEDOIDS
# ============================================================

mejor_k = int(
    tabla_k.loc[
        tabla_k["Silueta"].idxmax(),
        "K"
    ]
)

print("\nNÚMERO DE CLUSTERS SELECCIONADO:", mejor_k)

modelo_final = KMedoids(
    n_clusters=mejor_k,
    metric="euclidean",
    method="pam",
    init="k-medoids++",
    random_state=42
)

nc = modelo_final.fit_predict(XD_escalado)

# Agregar cluster a cada paciente
XDB2 = XDB.copy()

XDB2["Cluster"] = nc

print("\nBASE CON CLUSTERS")
display(XDB2.head())


# ============================================================
# 5. IDENTIFICAR LOS MEDOIDES
# ============================================================

medoides = XDB.iloc[
    modelo_final.medoid_indices_
].copy()

medoides["Cluster"] = range(mejor_k)

print("\nPACIENTES REPRESENTATIVOS DE CADA CLUSTER")
display(
    medoides.sort_values("Cluster")
)


# ============================================================
# 6. PERFIL GENERAL DE LOS CLUSTERS
# ============================================================

perfil = XDB2.groupby("Cluster").agg(
    Pacientes=("age", "size"),
    Edad_Promedio=("age", "mean"),
    Edad_Minima=("age", "min"),
    Edad_Maxima=("age", "max"),
    BMI_Promedio=("bmi", "mean"),
    Hijos_Promedio=("children", "mean"),
    Costo_Promedio=("charges", "mean"),
    Costo_Mediano=("charges", "median"),
    Costo_Total=("charges", "sum")
)

perfil["Porcentaje_Pacientes"] = (
    perfil["Pacientes"] / len(XDB2) * 100
)

perfil["Porcentaje_Costo_Total"] = (
    perfil["Costo_Total"] / XDB2["charges"].sum() * 100
)

print("\nCARACTERIZACIÓN DE LOS CLUSTERS")
display(perfil.round(2))


# ============================================================
# 7. DISTRIBUCIONES PORCENTUALES
# ============================================================

# Clasificar el costo en cuatro niveles
XDB2["Nivel_Costo"] = pd.qcut(
    XDB2["charges"],
    q=4,
    labels=["Bajo", "Medio-bajo", "Medio-alto", "Alto"]
)

# Composición de cada cluster
variables = [
    "smoker",
    "sex",
    "region",
    "Nivel_Costo"
]

for variable in variables:

    tabla = pd.crosstab(
        XDB2["Cluster"],
        XDB2[variable],
        normalize="index"
    ) * 100

    print(f"\nDISTRIBUCIÓN DE {variable.upper()} POR CLUSTER")
    display(tabla.round(2))


# ============================================================
# 8. ANÁLISIS TERRITORIAL
# ============================================================

# Cantidad de pacientes por región y cluster
conteo_region = pd.crosstab(
    XDB2["region"],
    XDB2["Cluster"]
)

# Porcentaje de cada cluster DENTRO de cada región
porcentaje_region = pd.crosstab(
    XDB2["region"],
    XDB2["Cluster"],
    normalize="index"
) * 100

# Costo promedio por región y cluster
costo_region = XDB2.pivot_table(
    index="region",
    columns="Cluster",
    values="charges",
    aggfunc="mean"
)

print("\nPACIENTES POR REGIÓN Y CLUSTER")
display(conteo_region)

print("\nDISTRIBUCIÓN PORCENTUAL DE CLUSTERS DENTRO DE CADA REGIÓN")
display(porcentaje_region.round(2))

print("\nCOSTO PROMEDIO POR REGIÓN Y CLUSTER")
display(costo_region.round(2))


# ============================================================
# 9. GRÁFICOS
# ============================================================

# GRÁFICO 1: Número de pacientes por cluster

XDB2["Cluster"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(8, 4),
    title="Número de pacientes por cluster"
)

plt.xlabel("Cluster")
plt.ylabel("Número de pacientes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# GRÁFICO 2: Costo promedio por cluster

XDB2.groupby("Cluster")["charges"].mean().plot(
    kind="bar",
    figsize=(8, 4),
    title="Costo promedio por cluster"
)

plt.xlabel("Cluster")
plt.ylabel("Costo promedio")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# GRÁFICO 3: Distribución territorial

porcentaje_region.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    title="Distribución porcentual de clusters por región"
)

plt.xlabel("Región")
plt.ylabel("Porcentaje de pacientes")
plt.xticks(rotation=0)
plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()


# GRÁFICO 4: Nivel de costo por cluster

(pd.crosstab(
    XDB2["Cluster"],
    XDB2["Nivel_Costo"],
    normalize="index"
) * 100).plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    title="Distribución del nivel de costo por cluster"
)

plt.xlabel("Cluster")
plt.ylabel("Porcentaje de pacientes")
plt.xticks(rotation=0)
plt.legend(
    title="Nivel de costo",
    loc="upper left"
)
plt.tight_layout()
plt.show()


# GRÁFICO 5: Fumadores por cluster

(pd.crosstab(
    XDB2["Cluster"],
    XDB2["smoker"],
    normalize="index"
) * 100).plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    title="Distribución de fumadores por cluster"
)

plt.xlabel("Cluster")
plt.ylabel("Porcentaje de pacientes")
plt.xticks(rotation=0)
plt.legend(
    title="Fumador",
    loc="upper left"
)
plt.tight_layout()
plt.show()

# Analisis


**Análisis de resultados – K-Medoids**

La aplicación de K-Medoids permitió identificar cuatro clústeres de pacientes, diferenciados principalmente por su condición de fumador. Los clústeres 0 y 2 están conformados exclusivamente por fumadores y presentan costos promedio de USD 30.679 y USD 33.042, respectivamente. En contraste, los clústeres 1 y 3 agrupan pacientes no fumadores, con costos promedio considerablemente inferiores, de USD 8.762 y USD 8.087.

Entre los grupos de fumadores, el clúster 2 registra un IMC promedio superior al del clúster 0 (31,50 frente a 29,61), acompañado de un mayor costo promedio. Las edades promedio son similares entre los cuatro grupos, situándose aproximadamente entre los 38 y 40 años.

**Distribución territorial**

La distribución porcentual de los clústeres es relativamente homogénea entre las cuatro regiones analizadas: noreste, noroeste, sureste y suroeste.

En todas las regiones predominan los clústeres 1 y 3, correspondientes a pacientes no fumadores. Los clústeres 0 y 2 representan una proporción menor, cercana al 20 % de los pacientes de cada región en conjunto.

Esto indica que los perfiles asociados con mayores costos no se concentran exclusivamente en una región, sino que están presentes en todo el territorio analizado.

**Conclusión y aplicación empresarial**

Los resultados permiten identificar una asociación entre el tabaquismo y los mayores costos registrados en la base de datos. Para la aseguradora, esta segmentación puede servir como punto de partida para diseñar programas preventivos diferenciados, especialmente iniciativas de cesación del tabaquismo y seguimiento de factores como el IMC.

Dado que los perfiles se encuentran distribuidos de manera relativamente uniforme entre las regiones, los resultados sugieren considerar una estrategia preventiva de cobertura nacional, ajustando los recursos según la cantidad de pacientes y los costos observados en cada territorio.

Limitación importante: el costo registrado no equivale directamente a gravedad clínica, y la asociación identificada no demuestra por sí sola una relación causal. Además, los cuatro grupos deben justificarse con el resultado del coeficiente de silueta obtenido en el modelo.